# Chained extend on LeiData wav1

Views results from [`scripts/run_extend_dictionary_leidata_wav1.sh`](../scripts/run_extend_dictionary_leidata_wav1.sh):

| Stage spans | `n_components` |
|---|---|
| 1, 2 | K=1 |
| 4, 8, 16, 30 | K=10 |

Loads `stage_wN.json` + each stage’s stamped `manifest.json`. Re-run cells after the shell script finishes.

```bash
bash analysis/dict_learning/scripts/run_extend_dictionary_leidata_wav1.sh
# log: …/wav1_extend_chain/run.log
```

In [ ]:
%matplotlib inline

import json
import re
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

REPO = Path("/home/yfreund/projects/gplrev3-webserver")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

LIBRARY_ROOT = Path("/home/yfreund/whale_processing")
CHAIN_ROOT = Path(
    LIBRARY_ROOT / "result/LeiData/dict_learning_templates/wav1_extend_chain"
)
DB_PATH = LIBRARY_ROOT / "result/dict_learning_catalog.sqlite"
WAV = LIBRARY_ROOT / "LeiData/SOCAL_H_76_230427_003345.x.wav"
STAGE_RE = re.compile(r"^stage_w(\d+)\.json$")


def _load_stage(path: Path) -> dict | None:
    if path.stat().st_size == 0:
        return None
    try:
        return json.loads(path.read_text())
    except json.JSONDecodeError:
        return None


def load_chain_stages(chain_root: Path) -> list[dict]:
    rows = []
    for path in sorted(chain_root.glob("stage_w*.json"), key=lambda p: p.name):
        m = STAGE_RE.match(path.name)
        if not m:
            continue
        summary = _load_stage(path)
        if summary is None:
            print(f"skip incomplete {path.name}")
            continue
        n_windows = int(m.group(1))
        out_dir = Path(summary["out_dir"])
        man_path = out_dir / "manifest.json"
        man = json.loads(man_path.read_text()) if man_path.is_file() else {}
        params = summary.get("params") or man.get("params") or {}
        e0 = man.get("energy_before")
        e1 = man.get("energy_after")
        frac = None
        if e0 is not None and e1 is not None and float(e0) > 0:
            frac = (float(e0) - float(e1)) / float(e0)
        rows.append(
            {
                "n_windows": n_windows,
                "summary": summary,
                "manifest": man,
                "out_dir": out_dir,
                "n_components": int(params.get("n_components") or man.get("n_components") or 1),
                "coverage": float(summary.get("coverage") or 0.0),
                "n_atoms": int(summary.get("n_atoms") or len(man.get("templates") or [])),
                "n_atoms_added": int(summary.get("n_atoms_added") or 0),
                "stop_reason": summary.get("stop_reason"),
                "dictionary_id": summary.get("dictionary_id"),
                "energy_before": e0,
                "energy_after": e1,
                "fraction_removed": frac,
                "n_matchings": len(man.get("matchings") or []),
            }
        )
    rows.sort(key=lambda r: r["n_windows"])
    return rows


stages = load_chain_stages(CHAIN_ROOT)
print(f"Chain root: {CHAIN_ROOT}")
print(f"Completed stages: {[s['n_windows'] for s in stages]}")
log_path = CHAIN_ROOT / "run.log"
if log_path.is_file():
    print(f"Run log: {log_path} ({log_path.stat().st_size} bytes)")

## Stage summary

In [ ]:
if not stages:
    raise FileNotFoundError(
        f"No complete stage_wN.json under {CHAIN_ROOT}. "
        "Wait for the shell script or check run.log."
    )

lines = [
    "| n_windows | K | coverage | n_atoms | added | matchings | %E removed | stop | dict_id |",
    "|---|---:|---:|---:|---:|---:|---:|---|---|",
]
for s in stages:
    pct = "—" if s["fraction_removed"] is None else f"{100*s['fraction_removed']:.1f}%"
    lines.append(
        f"| {s['n_windows']} | {s['n_components']} | {s['coverage']:.3f} | "
        f"{s['n_atoms']} | {s['n_atoms_added']} | {s['n_matchings']} | {pct} | "
        f"`{s['stop_reason']}` | {s['dictionary_id']} |"
    )
display(Markdown("\n".join(lines)))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
xs = [s["n_windows"] for s in stages]
axes[0].plot(xs, [s["coverage"] for s in stages], "o-")
axes[0].axhline(0.9, color="gray", ls="--", lw=1, label="min_coverage")
axes[0].set_xlabel("n_windows"); axes[0].set_ylabel("time-bin coverage"); axes[0].legend()
axes[0].set_title("Coverage vs span")

axes[1].plot(xs, [s["n_atoms"] for s in stages], "o-", label="total")
axes[1].plot(xs, [s["n_atoms_added"] for s in stages], "s--", label="added this stage")
axes[1].set_xlabel("n_windows"); axes[1].set_ylabel("atoms"); axes[1].legend()
axes[1].set_title("Dictionary size")

fracs = [s["fraction_removed"] for s in stages]
if any(f is not None for f in fracs):
    axes[2].plot(xs, [100 * (f or 0) for f in fracs], "o-")
axes[2].set_xlabel("n_windows"); axes[2].set_ylabel("% energy removed")
axes[2].set_title("Energy removed")
for ax in axes:
    ax.set_xscale("log", base=2)
    ax.set_xticks(xs)
    ax.set_xticklabels(xs)
fig.tight_layout()
plt.show()

## Templates (leading mode)

Pick a stage with the slider index below (default: last completed). For K=10 atoms, plot mode 0 of the subspace.

In [ ]:
STAGE_INDEX = -1  # last completed stage
stage = stages[STAGE_INDEX]
man = stage["manifest"]
out_dir = stage["out_dir"]
print(
    f"Showing n_windows={stage['n_windows']} K={stage['n_components']} "
    f"atoms={stage['n_atoms']}  {out_dir}"
)

templates = []
for item in man.get("templates") or []:
    arr = np.load(out_dir / item["file"])
    if arr.ndim == 3:
        view = arr[0]  # leading PCA mode
    else:
        view = arr
    templates.append((int(item["template_id"]), view, arr.ndim))

n = len(templates)
ncols = min(5, max(n, 1))
nrows = int(np.ceil(n / ncols)) if n else 1
fig, axes = plt.subplots(nrows, ncols, figsize=(2.4 * ncols, 2.2 * nrows), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
for i, (tid, view, ndim) in enumerate(templates):
    ax = axes[i // ncols][i % ncols]
    ax.imshow(view, aspect="auto", origin="lower", cmap="magma")
    ax.set_title(f"id={tid}" + (" (mode0)" if ndim == 3 else ""), fontsize=9)
    ax.axis("off")
fig.suptitle(f"Templates · w={stage['n_windows']} · K={stage['n_components']}", y=1.02)
fig.tight_layout()
plt.show()

counts = Counter(m["template_id"] for m in man.get("matchings") or [])
if counts:
    tids = sorted(counts)
    fig, ax = plt.subplots(figsize=(max(4, 0.35 * len(tids)), 3))
    ax.bar([str(t) for t in tids], [counts[t] for t in tids])
    ax.set_xlabel("template_id"); ax.set_ylabel("# matchings")
    ax.set_title("Matchings per atom")
    fig.tight_layout()
    plt.show()

## Window-1 matching overlay (first vs last stage)

Load the first spectrogram window from the catalog and draw matching intervals from the earliest and latest completed stages.

In [ ]:
from database import catalog


def _first_window_array(db_path: Path, wav: Path) -> tuple[np.ndarray, str]:
    with catalog.connect(db_path) as conn:
        wav_row = catalog.find_wav_file(conn, str(wav))
        if wav_row is None:
            raise FileNotFoundError(wav)
        segs = catalog.list_gpl_segmentations_for_wav(conn, wav_row["wav_id"])
        wins = catalog.list_windows_for_segmentation(
            conn, segs[0]["segmentation_id"], limit=1, offset=0
        )
        npy = Path(wins[0]["npy_path"])
        return np.load(npy), wins[0]["window_id"]


def _template_widths(man: dict) -> dict[int, int]:
    widths = {}
    for t in man.get("templates") or []:
        shape = t.get("shape") or []
        if len(shape) >= 2:
            widths[int(t["template_id"])] = int(shape[-1])
        else:
            widths[int(t["template_id"])] = int(t.get("width") or 1)
    return widths


spec, wid = _first_window_array(DB_PATH, WAV)
print(f"Window: {wid}  shape={spec.shape}")

pair = [stages[0], stages[-1]] if len(stages) >= 2 else stages[:1]
fig, axes = plt.subplots(len(pair), 1, figsize=(14, 3.2 * len(pair)), sharex=True)
if len(pair) == 1:
    axes = [axes]
cmap = plt.cm.tab20

for ax, s in zip(axes, pair):
    ax.imshow(spec, aspect="auto", origin="lower", cmap="gray")
    widths = _template_widths(s["manifest"])
    drawn = 0
    for m in s["manifest"].get("matchings") or []:
        if m.get("window_id") != wid:
            continue
        tid = int(m["template_id"])
        t0 = int(m["t_start"])
        w = widths.get(tid, 1)
        ax.axvspan(t0, t0 + w, color=cmap(tid % 20), alpha=0.35)
        drawn += 1
    ax.set_title(
        f"w={s['n_windows']} K={s['n_components']} · {drawn} matchings on window 1 · "
        f"coverage={s['coverage']:.3f}"
    )
    ax.set_ylabel("freq")
axes[-1].set_xlabel("time bin")
fig.tight_layout()
plt.show()

## Per-stage out dirs

Useful when chaining subspace stages via `--input-dictionary-dir`.

In [ ]:
for s in stages:
    print(
        f"w={s['n_windows']:2d}  K={s['n_components']:2d}  "
        f"cov={s['coverage']:.3f}  atoms={s['n_atoms']:3d}  {s['out_dir']}"
    )